# Module 2 Demo — Data Ingestion & Data Validation

**The Golden Rule of ML Engineering:** *Garbage In, Garbage Out — silently!*
An ML algorithm will happily train on negative ages, duplicate customers, or `"ultra_vip"` typos without crashing, and then produce terrible predictions in production.

### What You Will Learn in This Demo
1. **Multi-Source Data Ingestion**: Loading a CSV (`customers_raw.csv`) + JSON (`usage_events.json`), normalizing text/casing, deduplicating by primary key, and merging cleanly.
2. **Schema & Data Quality Validation**: Writing an automated **Data Contract** (required columns, types, allowed categories, numeric ranges, missingness threshold) using Pandas + Pydantic.
3. **Fail-Fast vs. Quarantine**: Running our validator on both a **healthy batch** and a **corrupted batch** (`customers_corrupted_batch.csv`) and saving a structured JSON validation report.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import pandas as pd
from pydantic import BaseModel, Field, ValidationError

RAW_CUSTOMERS_PATH = Path("data/customers_raw.csv")
RAW_USAGE_PATH = Path("data/usage_events.json")
CORRUPTED_PATH = Path("data/customers_corrupted_batch.csv")


## Part 1: Multi-Source Data Ingestion

Real-world datasets rarely live in a single clean CSV. At StreamPulse:
- Billing & profile data comes from the CRM database export (`customers_raw.csv`).
- Product engagement & support tickets come from the events API (`usage_events.json`).

Let's inspect the raw messiness first, then write a clean `ingest_streampulse_data()` function.


In [ ]:
raw_customers = pd.read_csv(RAW_CUSTOMERS_PATH)
with open(RAW_USAGE_PATH, "r", encoding="utf-8") as f:
    raw_usage = pd.DataFrame(json.load(f))

print(f"Raw customers rows : {len(raw_customers)} (unique customer_id: {raw_customers['customer_id'].nunique()})")
print(f"Raw usage rows     : {len(raw_usage)} (unique customer_id: {raw_usage['customer_id'].nunique()})")
print("\nRaw plan_tier values (notice messy casing & spaces!):")
print(raw_customers["plan_tier"].value_counts(dropna=False))
print("\nRaw country values:")
print(raw_customers["country"].value_counts(dropna=False))


In [ ]:
def compute_file_sha256(path: Path) -> str:
    """Return a short SHA-256 fingerprint for data lineage tracking."""
    h = hashlib.sha256()
    h.update(path.read_bytes())
    return h.hexdigest()[:12]


def ingest_streampulse_data(customers_csv: Path, usage_json: Path) -> tuple[pd.DataFrame, dict]:
    """Load, standardize, deduplicate, and join StreamPulse customer + usage data."""
    cust_df = pd.read_csv(customers_csv)
    with open(usage_json, "r", encoding="utf-8") as f:
        usage_df = pd.DataFrame(json.load(f))

    raw_row_count = len(cust_df)

    # 1. Standardize column names (lowercase, stripped)
    cust_df.columns = [c.strip().lower() for c in cust_df.columns]
    usage_df.columns = [c.strip().lower() for c in usage_df.columns]

    # 2. Clean whitespace & casing on string categories (preserving NaN as NaN!)
    if "plan_tier" in cust_df.columns:
        cust_df["plan_tier"] = cust_df["plan_tier"].apply(
            lambda x: x.strip().lower() if isinstance(x, str) else x
        )
    if "country" in cust_df.columns:
        cust_df["country"] = cust_df["country"].apply(
            lambda x: x.strip().upper() if isinstance(x, str) else x
        )
    if "payment_method" in cust_df.columns:
        cust_df["payment_method"] = cust_df["payment_method"].apply(
            lambda x: x.strip().lower() if isinstance(x, str) else x
        )

    # 3. Deduplicate on primary key customer_id
    duplicates_removed = int(cust_df.duplicated(subset=["customer_id"]).sum())
    cust_df = cust_df.drop_duplicates(subset=["customer_id"], keep="first")

    # 4. Left join usage metrics onto customers (drop orphan usage IDs with no billing profile)
    merged_df = cust_df.merge(usage_df, on="customer_id", how="left", validate="one_to_one")

    metadata = {
        "ingested_at_utc": datetime.now(timezone.utc).isoformat(),
        "customers_source": str(customers_csv),
        "customers_sha256": compute_file_sha256(customers_csv),
        "usage_source": str(usage_json),
        "usage_sha256": compute_file_sha256(usage_json),
        "raw_customer_rows": raw_row_count,
        "duplicates_removed": duplicates_removed,
        "final_rows": len(merged_df),
        "final_columns": list(merged_df.columns),
    }
    return merged_df, metadata


ingested_df, ingest_meta = ingest_streampulse_data(RAW_CUSTOMERS_PATH, RAW_USAGE_PATH)
print("Ingestion Metadata:")
print(json.dumps(ingest_meta, indent=2))
print("\nCleaned plan_tier values:")
print(ingested_df["plan_tier"].value_counts(dropna=False))


## Part 2: Data Validation — Building an Automated "Airport Security Checkpoint"

Before any dataset enters feature engineering or model training, it must pass **Data Validation** checks:
1. **Schema Check**: Are all required columns present?
2. **Primary Key Check**: Are `customer_id` values non-null and unique?
3. **Missingness Budget**: Is the missing percentage per column below our threshold (e.g., 15%)?
4. **Range & Domain Rules**:
   - `18 <= age <= 100` (where not null)
   - `tenure_months >= 0`
   - `0 <= monthly_charges <= 100`
   - `plan_tier` in `{"basic", "standard", "premium"}`
   - `churned` in `{0, 1}`

Let's implement a clear, reusable validator function AND also show how **Pydantic** validates individual incoming JSON records at inference time!


In [ ]:
REQUIRED_COLUMNS = [
    "customer_id", "age", "plan_tier", "country", "payment_method",
    "auto_pay", "tenure_months", "monthly_charges", "churned",
]
ALLOWED_PLANS = {"basic", "standard", "premium"}
ALLOWED_COUNTRIES = {"US", "IN", "UK", "CA", "DE"}


def validate_customer_dataframe(df: pd.DataFrame, max_missing_ratio: float = 0.15) -> dict:
    """Run schema and data-quality checks. Returns a structured validation report."""
    errors = []
    warnings = []

    # Check 1: Required columns
    missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing_cols:
        errors.append(f"Missing required columns: {missing_cols}")
        return {"passed": False, "row_count": len(df), "errors": errors, "warnings": warnings}

    # Check 2: Primary key uniqueness & non-null
    null_ids = int(df["customer_id"].isna().sum())
    dup_ids = int(df["customer_id"].duplicated().sum())
    if null_ids > 0:
        errors.append(f"Found {null_ids} null customer_id values.")
    if dup_ids > 0:
        errors.append(f"Found {dup_ids} duplicate customer_id values.")

    # Check 3: Missingness threshold per column
    for col in REQUIRED_COLUMNS:
        ratio = float(df[col].isna().mean())
        if ratio > max_missing_ratio:
            errors.append(f"Column '{col}' missing ratio {ratio:.1%} exceeds limit {max_missing_ratio:.1%}.")
        elif ratio > 0:
            warnings.append(f"Column '{col}' has {ratio:.1%} missing values (will be imputed in pipeline).")

    # Check 4: Numeric range checks (ignoring NaNs which are checked above)
    bad_age = df["age"].dropna().pipe(lambda s: ~s.between(18, 100)).sum()
    if bad_age > 0:
        errors.append(f"Found {int(bad_age)} rows with age outside [18, 100].")

    bad_tenure = (df["tenure_months"].dropna() < 0).sum()
    if bad_tenure > 0:
        errors.append(f"Found {int(bad_tenure)} rows with negative tenure_months.")

    bad_charges = df["monthly_charges"].dropna().pipe(lambda s: ~s.between(0, 100)).sum()
    if bad_charges > 0:
        errors.append(f"Found {int(bad_charges)} rows with monthly_charges outside [0, 100].")

    # Check 5: Allowed categorical values
    bad_plans = set(df["plan_tier"].dropna().unique()) - ALLOWED_PLANS
    if bad_plans:
        errors.append(f"Unexpected plan_tier values: {sorted(bad_plans)}")

    bad_countries = set(df["country"].dropna().unique()) - ALLOWED_COUNTRIES
    if bad_countries:
        errors.append(f"Unexpected country values: {sorted(bad_countries)}")

    # Check 6: Target label integrity
    bad_targets = set(df["churned"].dropna().unique()) - {0, 1}
    if bad_targets:
        errors.append(f"Invalid churned target values: {sorted(bad_targets)}")

    return {
        "passed": len(errors) == 0,
        "row_count": int(len(df)),
        "error_count": len(errors),
        "errors": errors,
        "warnings": warnings,
    }


clean_report = validate_customer_dataframe(ingested_df)
print("=== Validation Report: Clean Ingested Dataset ===")
print(json.dumps(clean_report, indent=2))
assert clean_report["passed"] is True


In [ ]:
# Now test the EXACT same validator on the corrupted incoming batch!
corrupted_df = pd.read_csv(CORRUPTED_PATH)
corrupted_report = validate_customer_dataframe(corrupted_df)

print("=== Validation Report: Corrupted Batch ===")
print(json.dumps(corrupted_report, indent=2))
assert corrupted_report["passed"] is False

# Save validation report artifact for auditability
report_path = Path("artifacts/validation_report_demo.json")
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(corrupted_report, indent=2), encoding="utf-8")
print(f"\nSaved audit report to: {report_path}")


## Part 3: Bonus Preview — Row-Level Validation with Pydantic

When your model later serves live predictions via an API (Day 8 & Day 14), data arrives **one JSON record at a time**. Pydantic lets you enforce the exact same contract in 10 lines!


In [ ]:
class CustomerRecordSchema(BaseModel):
    customer_id: str
    age: float = Field(ge=18, le=100)
    plan_tier: str = Field(pattern="^(basic|standard|premium)$")
    tenure_months: float = Field(ge=0)
    monthly_charges: float = Field(ge=0, le=100)


good_payload = {
    "customer_id": "CUST-50001",
    "age": 31,
    "plan_tier": "standard",
    "tenure_months": 14,
    "monthly_charges": 15.99,
}
bad_payload = {
    "customer_id": "CUST-50002",
    "age": -5,
    "plan_tier": "ultra_vip",
    "tenure_months": 14,
    "monthly_charges": 999.0,
}

print("Valid payload accepted:", CustomerRecordSchema(**good_payload))
print("\nTesting invalid payload:")
try:
    CustomerRecordSchema(**bad_payload)
except ValidationError as exc:
    for err in exc.errors():
        print(f"  - Field '{err['loc'][0]}': {err['msg']} (got {err['input']!r})")


### Key Takeaway for Module 2
1. **Never overwrite raw files** — ingest into cleaned dataframes/tables and record source hashes.
2. **Validate before modeling** — catch missing columns, impossible ranges, and surprise categories *before* they corrupt your model.
3. **Next:** Open `03_exercise_data_ingestion_and_validation_starter.ipynb` for your first hands-on lab!
